# 04 - MCP com Agentes
> Agentes autonomos usando infraestrutura MCP

**Modulo:** `05_mcp` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import json

class AmbienteMCP:
    def __init__(self): self.srvs={}; self.tools=[]
    def conectar(self,nome,tools_dict):
        self.srvs[nome]=tools_dict
        for tn,(fn,desc,schema) in tools_dict.items():
            self.tools.append({'name':f'{nome}__{tn}','description':f'[{nome}] {desc}','input_schema':schema})
    def call(self,full_name,args):
        srv,tool=full_name.split('__',1)
        return self.srvs[srv][tool][0](**args)

TAREFAS={}
env=AmbienteMCP()
env.conectar('tarefas',{
    'criar':(lambda titulo: TAREFAS.update({len(TAREFAS)+1:titulo}) or f'Tarefa {len(TAREFAS)} criada',
             'Cria tarefa.',{'type':'object','properties':{'titulo':{'type':'string'}},'required':['titulo']}),
    'listar':(lambda: json.dumps(TAREFAS,ensure_ascii=False),
              'Lista tarefas.',{'type':'object','properties':{}})
})

def agente(q):
    msgs=[{'role':'user','content':q}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=env.tools,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':str(env.call(b.name,b.input))})
        msgs.append({'role':'user','content':res})

print(agente('Crie 3 tarefas: estudar MCP, testar agentes, fazer evals. Depois liste tudo.'))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
